# Open-Sourcing at Gigaton

## A Skeights Case Study

<img src="logo.png" width="300">

Alex Haslam

# What is Open Source?

**Open source** means the source code is publicly available. Anyone can read it, use it, and often modify it.

It's not just code. Open source can be:
- **Libraries:** reusable building blocks (e.g. scikit-learn for ML, pandas for data)
- **Tools:** software you run (e.g. VS Code, Linux)
- **Datasets, documentation, standards...**

Most modern software is built on top of open-source foundations.

![xkcd 2347](https://imgs.xkcd.com/comics/dependency.png)

*Source: [xkcd.com/2347](https://xkcd.com/2347/)*

# Why Should Companies Contribute?

- **Giving back:** we rely on open source, we should contribute to it
- **Technical brand:** shows we do interesting engineering work
- **Hiring signal:** candidates can see what we build and how we build it
- **Better code:** forces clean APIs, good docs, and proper tests

# The Problem: Storing ML Models

As part of the ML Refactor, we needed a way to save and load trained models.

The industry default for scikit-learn is **pickle**.

But what even is pickle?

Pickle serialises Python objects to bytes. When you load a pickle file, it **runs arbitrary Python code**.

Let's see why that's a problem...

# Demo: A Malicious Pickle

In [ ]:
import pickle


class MaliciousModel:
    """This looks like a normal model, but it runs code when loaded."""

    def __reduce__(self):
        # __reduce__ controls what happens when this object is unpickled.
        # Instead of just reconstructing the object, it runs print().
        # In real life, this could be: os.system("rm -rf /") or worse.
        return (print, ("\n>>> I HAVE A VIRUS! <<<\n\nThis could delete files, steal credentials, or install malware.\nAll from just loading a 'model' file.",))


# Save the malicious "model"
with open("/tmp/evil_model.pkl", "wb") as f:
    pickle.dump(MaliciousModel(), f)

print("Saved 'model' to evil_model.pkl")
print("Now let's load it...")

In [ ]:
# Loading the pickle runs the malicious code
with open("/tmp/evil_model.pkl", "rb") as f:
    model = pickle.load(f)  # This runs the code!

# Other Problems with Pickle

Security isn't the only issue:

- **Opaque:** you can't inspect a pickle file without loading it (which runs code!)
- **Fragile:** rename a class or move a module, and all your saved models break
- **Python-only:** can't read pickles from other languages

There is an existing alternative called **skops** (from the scikit-learn team). It fixes the security problem, but the saved files are still opaque binary blobs: you can't inspect the model config without deserialising it.

We wanted something where the model state is **structured data** you can read, diff, and version.

# Introducing Skeights

### Why "skeights"?

There's a convention in the scikit-learn ecosystem of using the **sk-** prefix for related libraries (sklearn, skops, sktime, skorch...).

So: **sk** + safe**tensors** + wei**ghts** = **skeights**. Pronounced "skates".

### How is it different? 

We save the model into two files:

```mermaid
flowchart TD
    A[Fitted model] -->|skeights.save| B["model.safetensors"]
    A -->|skeights.save| C["model.json"]
    B --- D["Numeric weights<br/>Binary, safe<br/>No code execution"]
    C --- E["Model config as JSON:<br/>{&quot;alpha&quot;: 0.1, &quot;fit_intercept&quot;: true}<br/>Human-readable, diffable"]
```

No pickle. No arbitrary code. You can read the JSON in a text editor.

Also supports **LightGBM** and **XGBoost**: close cousins of scikit-learn that provide specialised tree-based models. They all look the same to Python, so skeights handles them all.

# What is Safetensors?

**Hugging Face** is the biggest platform for sharing open-source ML models (essentially think GitHub but for ML). 

Models like Llama, Mistral, and other LLMs are all hosted there. 

They created **safetensors**: a binary format that can only store arrays of numbers. No code, no Python references, no way to sneak in malware.

<img src="huggingface.png" width="600">

This is now commonly used for sharing models. Skeights brings it to scikit-learn.

# Demo: Saving a Model with Skeights

In [ ]:
import json

import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

import skeights

# Load data and train a pipeline
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=200)),
])
pipe.fit(X_train, y_train)

print(f"Accuracy: {pipe.score(X_test, y_test):.1%}")

In [ ]:
import os

# Save with skeights
skeights.save(pipe, "/tmp/model.safetensors", "/tmp/model.json")

# Show the output files
st_size = os.path.getsize("/tmp/model.safetensors")
json_size = os.path.getsize("/tmp/model.json")

print("/tmp/")
print(f"  model.safetensors  ({st_size:,} bytes) - numeric weights")
print(f"  model.json         ({json_size:,} bytes) - model config")

# What's Inside the Files?

In [ ]:
# The safetensors file is binary: just numbers, no code
# If you try to read it as text:
with open("/tmp/model.safetensors", "rb") as f:
    raw = f.read(200)
print(raw)

Looks like nonsense! But that's the point: it's just numbers stored efficiently. 

There is no Python code hiding inside.

In [ ]:
# The JSON file is human-readable: just config, no code
with open("/tmp/model.json") as f:
    config = json.load(f)

print(json.dumps(config, indent=2))

# Look at That!

- Every hyperparameter is right there in plain JSON
- It is easier to build tools to create new models (eg Claude)
- **No code runs when you read it**

# File Size Wins

For tree-based models, the native save formats store tree parameters as huge text strings.

Skeights stores them as typed numpy arrays in safetensors instead. Let's see the difference.

In [ ]:
import xgboost as xgb

# Train an XGBoost model to show the file size difference
xgb_model = xgb.XGBRegressor(n_estimators=100, max_depth=6, random_state=42)
xgb_model.fit(X_train, y_train)

# Save with native XGBoost format (JSON text)
xgb_model.save_model("/tmp/xgb_native.json")
native_size = os.path.getsize("/tmp/xgb_native.json")

# Save with skeights (safetensors binary)
skeights.save(xgb_model, "/tmp/xgb_skeights.safetensors", "/tmp/xgb_skeights.json")
skeights_size = os.path.getsize("/tmp/xgb_skeights.safetensors") + os.path.getsize("/tmp/xgb_skeights.json")

print(f"Native XGBoost format:  {native_size:>10,} bytes")
print(f"Skeights format:        {skeights_size:>10,} bytes")
print(f"Reduction:              {1 - skeights_size / native_size:>10.0%}")

In [ ]:
# Why is the native format so big? It stores tree parameters as text:
with open("/tmp/xgb_native.json") as f:
    native_json = f.read()

print("=== Native XGBoost JSON (first 500 chars) ===\n")
print(native_json[:500])
print("\n... thousands more lines of text like this")

In [ ]:
# Compare: the skeights JSON is clean and compact
with open("/tmp/xgb_skeights.json") as f:
    skeights_json = json.load(f)

print("=== Skeights JSON for the same XGBoost model ===\n")
print(json.dumps(skeights_json, indent=2)[:1000])

Similar results across other tree models:

| Model | Native format | Skeights | Reduction |
|-------|--------------|----------|-----------|
| XGBoost | 100% | ~15% | **85% smaller** |
| LightGBM | 100% | ~57-70% | **30-43% smaller** |

# The Open-Sourcing Process

The safetensors + JSON architecture came from **Bob** as part of the ML Refactor design.

I spotted that the serialisation code was completely generic: nothing Gigaton-specific about it. So we extracted it into its own package and linked it back in as a dependency.

<table><tr>
<td>

**Before**

```mermaid
flowchart TD
    subgraph carbo["carbo (internal codebase)"]
        A["ML pipeline code"]
        B["serialisation code"]
        A --- B
    end
```

</td>
<td>

**After**

```mermaid
flowchart TD
    subgraph carbo["carbo (internal codebase)"]
        C["ML pipeline code"]
    end
    D["skeights<br/>(open-source on PyPI)"]
    C -->|"pip install skeights"| D
```

</td>
</tr></table>

# Step 1: Spotting the Opportunity

Not everything should be open-sourced. Good candidates are:

- **Generic:** solves a problem others have too
- **Self-contained:** no internal dependencies
- **No secret sauce:** not a competitive advantage

Skeights ticked all three boxes.

After separating it out, I was able to improve it independently from carbo:
- added LightGBM and XGBoost support
- extended test coverage plus CI
- wrote proper docs
- added tests running across matrix of Python and sklearn versions to make sure it is robust

Hopefully others contribute and make it better too.

# Step 2: Publishing

- Published to **PyPI** (the Python Package Index): anyone in the world can install it with `pip install skeights`
- We have a **Gigaton organisation on PyPI**, so future packages can go under the same umbrella
- There's a **GitHub Actions workflow** that publishes automatically when we tag a release
- Source code publicly available on **GitHub**

<img src="pypi.png" width="600">

# Step 3: Getting It Out There

A repo sitting on GitHub isn't enough if people don't know it exists.

- Wrote a [blog post](https://alxhslm.github.io/projects/skeights/) explaining the why and how
- Posted on **Hacker News**: no traction, these things are hit or miss
- Shared on **LinkedIn**: decent traction, 1,100+ impressions and 27 likes
- **3 GitHub stars** so far: not exactly trending, but that's fine

The goal isn't to go viral. For us it's about the technical brand and giving back, not building the next big open-source project.

<img src="hackernews.png" width="500"> <img src="linkedin.png" width="350">

# What's Next?

- Are there other pieces of internal code that could be open-sourced?
- Think about what you work on that is **generic**, **self-contained**, and **not our secret sauce**
- It doesn't have to be a library: could be a tool, a dataset, a blog post

If you have ideas, I'm happy to help!

# Thanks!

- **Repo:** [github.com/carbon-re/skeights](https://github.com/carbon-re/skeights)
- **Blog post:** [alxhslm.github.io/projects/skeights/](https://alxhslm.github.io/projects/skeights/)
- **Install:** `pip install skeights`

Questions?